# Lab 7: Programowanie Obiektowe (OOP) w Fizyce

## 1. Od podejścia proceduralnego do obiektowego – po co fizykowi klasy?

W programowaniu proceduralnym dane (np. masa, ładunek, położenie, prędkość) oraz funkcje operujące na tych danych istnieją osobno. Przy symulacji układu wielu ciał zarządzanie dziesiątkami oddzielnych list staje się trudne i podatne na błędy.

**Programowanie obiektowe (OOP – Object-Oriented Programming)** pozwala połączyć **dane (atrybuty)** oraz **operacje na tych danych (metody)** w jedną spójną całość:
* **Klasa (`class`)** to szablon lub przepis teoretyczny (np. ogólny model cząstki naładowanej).
* **Obiekt (instancja)** to konkretny egzemplarz utworzony na podstawie klasy (np. konkretny elektron lub proton o zadanych parametrach).

> *Uwaga:* Z obiektów korzystasz w Pythonie od pierwszych zajęć – każda lista (`pomiary.append()`), tablica NumPy (`dane.mean()`) czy oś wykresu (`ax.plot()`) to obiekt łączący dane z wbudowanymi metodami.

---
## 2. Anatomia klasy: konstruktor `__init__`, `self` oraz atrybuty

* Klasę definiujemy słowem kluczowym `class` (nazwy klas piszemy z wielkiej litery, tzw. *CamelCase*).
* Metoda specjalna `__init__` to **konstruktor** – uruchamia się automatycznie w momencie tworzenia nowego obiektu i nadaje mu początkowe cechy.
* Parametr `self` oznacza **konkretny, bieżący obiekt**, na którym właśnie pracujemy.

In [11]:
class Czastka:
    def __init__(self, nazwa, masa, ladunek, x=0.0, v=0.0):
        # Atrybuty obiektu (stan fizyczny cząstki w układzie SI)
        self.nazwa = nazwa
        self.masa = masa        # [kg]
        self.ladunek = ladunek  # [C]
        self.x = x              # położenie [m]
        self.v = v              # prędkość [m/s]

# Tworzenie konkretnych obiektów (instancji)
elektron = Czastka("Elektron", masa=9.11e-31, ladunek=-1.602e-19, x=0.0, v=1.5e6)
proton = Czastka("Proton", masa=1.67e-27, ladunek=1.602e-19, x=0.05, v=0.0)

# Odczyt atrybutów za pomocą kropki
print(f"Obiekt: {elektron.nazwa}, masa: {elektron.masa:.2e} kg, prędkość: {elektron.v:.2e} m/s")
print(f"Obiekt: {proton.nazwa}, położenie początkowe: {proton.x} m")

Obiekt: Elektron, masa: 9.11e-31 kg, prędkość: 1.50e+06 m/s
Obiekt: Proton, położenie początkowe: 0.05 m


---
## 3. Metody – zachowanie i ewolucja układu w czasie

**Metoda** to funkcja zdefiniowana wewnątrz klasy (zawsze przyjmuje `self` jako pierwszy argument).
Metody dzielimy zazwyczaj na:
1. **Obliczające wartości** na podstawie stanu obiektu (zwracające wynik przez `return`).
2. **Modyfikujące stan** obiektu (np. przesunięcie cząstki w kroku czasowym $\Delta t$ pod wpływem jednorodnego pola elektrycznego $E$).

In [14]:
class Czastka:
    def __init__(self, nazwa, masa, ladunek, x=0.0, v=0.0):
        self.nazwa = nazwa
        self.masa = masa
        self.ladunek = ladunek
        self.x = x
        self.v = v

    def energia_kinetyczna(self):
        # Metoda zwracająca aktualną energię kinetyczną [J]
        return 0.5 * self.masa * self.v**2

    def ruch_w_polu_E(self, E, dt):
        # Metoda aktualizująca stan cząstki po czasie dt w polu elektrycznym E [V/m]
        # Siła Coulomba F = q * E, przyspieszenie a = F / m
        a = (self.ladunek * E) / self.masa
        self.v += a * dt
        self.x += self.v * dt

elektron = Czastka("Elektron", 9.11e-31, -1.602e-19, x=0.0, v=1.0e6)
print(f"Ek początkowa: {elektron.energia_kinetyczna():.2e} J")

# Symulacja kroku czasowego dt = 1 ns w polu przyspieszającym E = -500 V/m
elektron.ruch_w_polu_E(E=-500, dt=1e-9)
print(f"Nowe położenie: {elektron.x:.5f} m, nowa prędkość: {elektron.v:.2e} m/s")
print(f"Ek końcowa:     {elektron.energia_kinetyczna():.2e} J")

Ek początkowa: 4.56e-19 J
Nowe położenie: 0.00109 m, nowa prędkość: 1.09e+06 m/s
Ek końcowa:     5.39e-19 J


---
## 4. Metody specjalne (`__str__` i przeciążanie operatorów)

Metody otoczone podwójnym podkreśleniem (tzw. *dunder methods*) pozwalają zintegrować własne klasy z wbudowanymi funkcjami i operatorami Pythona:
* `__str__(self)` – definiuje tekstową reprezentację obiektu wywoływaną przez funkcję `print()`.
* `__add__(self, inny)` – definiuje zachowanie operatora `+` dla dwóch obiektów tej klasy.

In [17]:
import math

class Wektor2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def dlugosc(self):
        return math.hypot(self.x, self.y)

    def __add__(self, inny):
        # Pozwala dodawać dwa obiekty Wektor2D za pomocą znaku '+'
        return Wektor2D(self.x + inny.x, self.y + inny.y)

    def __str__(self):
        # Czytelny opis podczas wywołania print()
        return f"[{self.x:.2f}, {self.y:.2f}]"

F1 = Wektor2D(3.0, 4.0)
F2 = Wektor2D(1.0, -2.0)
F_wypadkowa = F1 + F2

print("Siła F1:        ", F1)
print("Siła F2:        ", F2)
print("Siła wypadkowa: ", F_wypadkowa)
print(f"Wartość siły wypadkowej: {F_wypadkowa.dlugosc():.2f} N")

Siła F1:         [3.00, 4.00]
Siła F2:         [1.00, -2.00]
Siła wypadkowa:  [4.00, 2.00]
Wartość siły wypadkowej: 4.47 N


## 5. Dziedziczenie (Inheritance) – hierarchia modeli fizycznych

Dziedziczenie pozwala tworzyć nowe klasy (**klasy pochodne**) na bazie istniejących (**klas bazowych**). Klasa pochodna automatycznie przejmuje wszystkie atrybuty i metody klasy nadrzędnej, a funkcja `super()` pozwala wywołać konstruktor klasy bazowej.

In [23]:
class Elektron(Czastka):
    def __init__(self, x=0.0, v=0.0):
        # Przekazanie znanych stałych fizycznych do konstruktora klasy bazowej Czastka
        super().__init__("Elektron", masa=9.11e-31, ladunek=-1.602e-19, x=x, v=v)

class Proton(Czastka):
    def __init__(self, x=0.0, v=0.0):
        super().__init__("Proton", masa=1.67e-27, ladunek=1.602e-19, x=x, v=v)

e1 = Elektron(x=0.01, v=2.0e6)
p1 = Proton(x=0.0, v=5.0e4)

print(f"{e1.nazwa}: masa = {e1.masa:.2e} kg, Ek = {e1.energia_kinetyczna():.2e} J")
print(f"{p1.nazwa}:   masa = {p1.masa:.2e} kg, Ek = {p1.energia_kinetyczna():.2e} J")

Elektron: masa = 9.11e-31 kg, Ek = 1.82e-18 J
Proton:   masa = 1.67e-27 kg, Ek = 2.09e-18 J


---
## ZADANIA DO SAMODZIELNEGO ROZWIĄZANIA

### Zadanie 1: Klasa `Opornik` (Prawo Ohma i moc wydzielana)
Napisz klasę `Opornik`, która reprezentuje element obwodu elektrycznego:
1. Konstruktor `__init__(self, rezystancja, moc_max)` zapisuje opór elektryczny $R$ [$\Omega$] oraz dopuszczalną moc maksymalną $P_{\max}$ [W].
2. Metoda `natezenie(self, U)` zwraca natężenie prądu płynącego przez opornik przy przyłożonym napięciu $U$ zgodnie z prawem Ohma: $I = \frac{U}{R}$.
3. Metoda `moc(self, U)` zwraca wydzielaną moc cieplną: $P = \frac{U^2}{R}$.
4. Metoda `czy_przepalony(self, U)` zwraca wartość logiczną `True`, jeśli moc wydzielana przy napięciu $U$ przekracza `moc_max`, a w przeciwnym razie `False`.

**Test:** Utwórz opornik o parametrach $R = 100\ \Omega$ oraz $P_{\max} = 0.25\ \text{W}$. Sprawdź jego działanie dla napięć $U = 3\ \text{V}$ oraz $U = 6\ \text{V}$.

In [54]:
# Twój kod tutaj


### Zadanie 2: Klasa `WahadloMatematyczne`
Zaprojektuj klasę `WahadloMatematyczne`:
1. Konstruktor `__init__(self, dlugosc, g=9.81)` przyjmuje długość nici $l$ [m] oraz przyspieszenie grawitacyjne $g$ [$	ext{m/s}^2$] (z wartością domyślną `9.81`).
2. Metoda `okres(self)` oblicza i zwraca okres małych drgań: $T = 2\pi \sqrt{\frac{l}{g}}$.
3. Metoda `czestotliwosc(self)` zwraca częstotliwość drgań: $f = \frac{1}{T}$.
4. Metoda `energia_calkowita(self, masa, amplituda)` oblicza całkowitą energię mechaniczną wahadła dla zadanej masy $m$ [kg] oraz amplitudy liniowej $A$ [m] ze wzoru: $E = \frac{1}{2} m \omega^2 A^2$, gdzie $\omega = \frac{2\pi}{T}$.

**Test:** Porównaj okres i częstotliwość tego samego wahadła o długości $l = 1.0\ \text{m}$ na Ziemi ($g = 9.81\ \text{m/s}^2$) oraz na Księżycu ($g = 1.62\ \text{m/s}^2$).

In [57]:
import math

# Twój kod tutaj:



### Zadanie z gwiazdką: Symulacja wiązki cząstek (OOP + Listy + Matplotlib)
Wykorzystaj zdefiniowaną wcześniej klasę `Elektron` (dziedziczącą po `Czastka` z metodą `ruch_w_polu_E`):
1. Utwórz listę 4 obiektów klasy `Elektron`, startujących z położenia $x = 0\ \text{m}$ z różnymi prędkościami początkowymi: $v_0 \in [1.0\times 10^6,\ 2.0\times 10^6,\ 3.0\times 10^6,\ 4.0\times 10^6]\ \text{m/s}$.
2. Przeprowadź symulację ich ruchu w jednorodnym polu elektrycznym hamującym $E = 2000\ \text{V/m}$ przez $200$ kroków czasowych o długości $\Delta t = 5 \times 10^{-11}\ \text{s}$ (łącznie $10\ \text{ns}$). Zapisuj historię położeń $x(t)$ każdego elektronu.
3. Wyrysuj trajektorie $x(t)$ wszystkich czterech elektronów na jednym wykresie w **Matplotlib** (z legendą, opisami osi i siatką).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Twój kod tutaj:

